# 03 — Modeling

Train and compare multiple classification models for loan default prediction using the SMOTE-balanced processed data.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
processed = joblib.load("../models/processed_data.pkl")

X_train = processed["X_train_smote"]
X_test  = processed["X_test"]
y_train = processed["y_train_smote"]
y_test  = processed["y_test"]

print("Processed Data Loaded Successfully")
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

Processed Data Loaded Successfully
X_train shape: (15468, 3)
X_test shape : (2000, 3)


In [3]:
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(random_state=42, n_jobs=-1),
    "Extra Trees":         ExtraTreesClassifier(random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
    "AdaBoost":            AdaBoostClassifier(random_state=42),
    "XGBoost":             XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=-1),
    "LightGBM":            LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost":            CatBoostClassifier(verbose=0, random_state=42),
}

In [4]:
results = []
trained_models = {}

for name, model in models.items():

    print("=" * 60)
    print(f"Training {name} ...")

    model.fit(X_train, y_train)
    trained_models[name] = model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append([
        name,
        round(accuracy_score(y_test, y_pred), 4),
        round(precision_score(y_test, y_pred, zero_division=0), 4),
        round(recall_score(y_test, y_pred, zero_division=0), 4),
        round(f1_score(y_test, y_pred, zero_division=0), 4),
        round(roc_auc_score(y_test, y_prob), 4),
    ])

    print(f"  ROC-AUC: {results[-1][5]}")

print("\nAll models trained!")

Training Logistic Regression ...
  ROC-AUC: 0.9488
Training Decision Tree ...
  ROC-AUC: 0.8096
Training Random Forest ...


  ROC-AUC: 0.915
Training Extra Trees ...
  ROC-AUC: 0.9154
Training Gradient Boosting ...


  ROC-AUC: 0.9328
Training AdaBoost ...
  ROC-AUC: 0.9382
Training XGBoost ...


  ROC-AUC: 0.9191
Training LightGBM ...


  ROC-AUC: 0.9264
Training CatBoost ...


  ROC-AUC: 0.9372

All models trained!


In [5]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"]
)

results_df.sort_values(by="ROC AUC", ascending=False).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.8565,0.1746,0.8806,0.2914,0.9488
1,AdaBoost,0.8550,0.1711,0.8657,0.2857,0.9382
2,CatBoost,0.8890,0.2031,0.7910,0.3232,0.9372
3,Gradient Boosting,0.8525,0.1647,0.8358,0.2752,0.9328
4,LightGBM,0.8860,0.1962,0.7761,0.3133,0.9264
5,XGBoost,0.8920,0.1959,0.7164,0.3077,0.9191
6,Extra Trees,0.8970,0.2068,0.7313,0.3224,0.9154
7,Random Forest,0.9025,0.2168,0.7313,0.3345,0.9150
8,Decision Tree,0.8965,0.2034,0.7164,0.3168,0.8096


In [6]:
best_model_name = results_df.sort_values(by="ROC AUC", ascending=False).iloc[0]["Model"]

print("Best Model:", best_model_name)

Best Model: Logistic Regression


In [7]:
best_model = trained_models[best_model_name]

joblib.dump(best_model, "../models/best_model.pkl")

print("Best Model Saved Successfully")

Best Model Saved Successfully


In [8]:
print(classification_report(
    y_test,
    best_model.predict(X_test),
    target_names=["No Default", "Default"]
))

              precision    recall  f1-score   support

  No Default       1.00      0.86      0.92      1933
     Default       0.17      0.88      0.29        67

    accuracy                           0.86      2000
   macro avg       0.58      0.87      0.61      2000
weighted avg       0.97      0.86      0.90      2000



In [9]:
cm = confusion_matrix(y_test, best_model.predict(X_test))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.savefig("../data/processed/confusion_matrix.png", dpi=80)
plt.show()
print("Plot saved.")

Plot saved.


In [10]:
preprocessor = joblib.load("../models/preprocessor.pkl")

feature_names = preprocessor.get_feature_names_out()

print("Feature names:", feature_names)

Feature names: ['num__Employed' 'num__Bank Balance' 'num__Annual Salary']


In [11]:
if hasattr(best_model, "feature_importances_"):

    importance = pd.DataFrame({
        "Feature":    feature_names,
        "Importance": best_model.feature_importances_
    }).sort_values(by="Importance", ascending=False)

    display(importance.head(20))
else:
    print(f"{best_model_name} does not expose feature_importances_.")

Logistic Regression does not expose feature_importances_.


In [12]:
if hasattr(best_model, "feature_importances_"):

    top20 = importance.head(20)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=top20, x="Importance", y="Feature")
    plt.title("Top Feature Importances")
    plt.tight_layout()
    plt.savefig("../data/processed/feature_importance.png", dpi=80)
    plt.show()
    print("Plot saved.")

In [13]:
results_df.to_csv("../models/model_comparison.csv", index=False)

print("Comparison Table Saved")

Comparison Table Saved


In [14]:
print("=" * 60)
print("Model Training Completed Successfully")
print("=" * 60)
print(results_df.sort_values(by="ROC AUC", ascending=False).to_string(index=False))

Model Training Completed Successfully
              Model  Accuracy  Precision  Recall  F1 Score  ROC AUC
Logistic Regression    0.8565     0.1746  0.8806    0.2914   0.9488
           AdaBoost    0.8550     0.1711  0.8657    0.2857   0.9382
           CatBoost    0.8890     0.2031  0.7910    0.3232   0.9372
  Gradient Boosting    0.8525     0.1647  0.8358    0.2752   0.9328
           LightGBM    0.8860     0.1962  0.7761    0.3133   0.9264
            XGBoost    0.8920     0.1959  0.7164    0.3077   0.9191
        Extra Trees    0.8970     0.2068  0.7313    0.3224   0.9154
      Random Forest    0.9025     0.2168  0.7313    0.3345   0.9150
      Decision Tree    0.8965     0.2034  0.7164    0.3168   0.8096
